# 🧠 Synthetic Dataset Creation — Groq Pipeline
### Real-Time Query Expansion & Topic Tagging — Stage 1

**Model:** `llama-4-scout-17b-16e-instruct` via Groq API  
**Target:** 500 annotated conversations  
**Pipeline:** Generate → Annotate → Validate  
**Saves:** CSV + JSON · Checkpoint every 50 samples · 3 retries on failure


## 0. Install

In [1]:
!pip install groq tqdm pandas scikit-learn -q
print('✅ done')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 6.7 MB/s eta 0:00:00
✅ done


## 1. Imports & Config

In [ ]:
import os, json, time, re, csv, math
from pathlib import Path
from datetime import datetime
from collections import Counter
import pandas as pd
from groq import Groq
from tqdm.notebook import tqdm

# ── CONFIG ───────────────────────────────────────────────────────
GROQ_API_KEY   = "GROQ_API_KEY"   # <── replace
MODEL          = "meta-llama/llama-4-scout-17b-16e-instruct"
TARGET_SAMPLES = 500
SAVE_EVERY     = 50
MAX_RETRIES    = 3
RETRY_DELAY    = 5
OUTPUT_DIR     = Path("dataset_output")
OUTPUT_DIR.mkdir(exist_ok=True)

CSV_PATH        = OUTPUT_DIR / "conversations.csv"
JSON_PATH       = OUTPUT_DIR / "conversations.json"
CHECKPOINT_PATH = OUTPUT_DIR / "checkpoint.json"

ALLOWED_L1 = {"Politics","Sports","Technology","Entertainment",
              "Health","History","Geography","General"}

client = Groq(api_key=GROQ_API_KEY)
print(f"✅ Groq client ready | Model: {MODEL}")
print(f"📁 Output → {OUTPUT_DIR.resolve()}")

✅ Groq client ready | Model: meta-llama/llama-4-scout-17b-16e-instruct
📁 Output → /content/dataset_output


## 2. Seed Pairs

In [3]:
BASE_SEEDS = [
    ("Politics","India","Narendra Modi"),("Politics","India","Rahul Gandhi"),
    ("Politics","India","Manmohan Singh"),("Politics","UK","Rishi Sunak"),
    ("Politics","UK","Boris Johnson"),("Politics","USA","Joe Biden"),
    ("Politics","USA","Donald Trump"),("Politics","USA","Barack Obama"),
    ("Sports","Cricket","Virat Kohli"),("Sports","Cricket","MS Dhoni"),
    ("Sports","Cricket","Rohit Sharma"),("Sports","Football","Lionel Messi"),
    ("Sports","Football","Cristiano Ronaldo"),("Sports","Tennis","Novak Djokovic"),
    ("Sports","Tennis","Rafael Nadal"),("Sports","Olympics","Neeraj Chopra"),
    ("Technology","AI","OpenAI"),("Technology","AI","Geoffrey Hinton"),
    ("Technology","AI","Google DeepMind"),("Technology","Space","ISRO"),
    ("Technology","Space","SpaceX"),("Technology","Gadgets","Apple iPhone"),
    ("Entertainment","Bollywood","Shah Rukh Khan"),("Entertainment","Bollywood","Aamir Khan"),
    ("Entertainment","Hollywood","Christopher Nolan"),("Entertainment","Hollywood","Marvel Studios"),
    ("Health","Nutrition","intermittent fasting"),("Health","Mental Health","anxiety"),
    ("Health","Fitness","yoga"),("History","India","Mahatma Gandhi"),
    ("History","World Wars","World War II"),("Geography","Countries","India"),
    ("Geography","Cities","Paris"),("General","General",""),
]

repeats   = math.ceil(TARGET_SAMPLES / len(BASE_SEEDS))
ALL_SEEDS = (BASE_SEEDS * repeats)[:TARGET_SAMPLES]
print(f"Base seeds: {len(BASE_SEEDS)} | Expanded: {len(ALL_SEEDS)}")
print("\nLabel distribution:")
for k, v in sorted(Counter(s[0] for s in ALL_SEEDS).items()):
    print(f"  {k:<18} {v:>4}")

Base seeds: 34 | Expanded: 500

Label distribution:
  Entertainment        58
  General              14
  Geography            28
  Health               42
  History              28
  Politics            120
  Sports              120
  Technology           90


## 3. Prompts (Compact)

In [4]:
# ── STAGE 1: CONVERSATION GENERATION ────────────────────────────
STAGE1_SYSTEM = """Generate a realistic 24-message chat (12 User + 12 Assistant, strictly alternating, User first) for NLP training.
Rules:
- Stay on the given TOPIC and ENTITY throughout.
- User messages must mix: direct questions, pronoun refs (his/her/their), elliptical follow-ups (what about X?), at least 1 interruption (brb/wait), resumption after break, a topic switch, and a comparison.
- Assistant replies: factual, concise, contextual.
- Output ONLY the conversation. No labels or extra text.
Format:
User: ...
Assistant: ...
(repeat, exactly 24 lines)

Example (Politics-India / Narendra Modi):
User: who is the pm of india?
Assistant: Narendra Modi has been Prime Minister since 2014.
User: what are his duties?
Assistant: He leads the Cabinet, sets policy, and represents India globally.
User: and internationally?
Assistant: He attends G20, UN summits and handles bilateral diplomacy.
User: wait a sec, brb
Assistant: Sure!
User: back — what about before him?
Assistant: Manmohan Singh was PM from 2004–2014.
User: same responsibilities?
Assistant: Yes, the PM's constitutional role stays the same.
User: who leads opposition?
Assistant: Rahul Gandhi leads the main opposition, the INC.
User: compare both
Assistant: Modi (BJP) focuses on growth; Rahul (INC) emphasises welfare.
User: got it thanks
Assistant: Happy to help!"""

def stage1_user_prompt(l1, l2, entity):
    e = entity or "general small talk"
    return f"Topic: {l1} - {l2}\nEntity: {e}\n\nGenerate the 20-message conversation now."


# ── STAGE 2: ANNOTATION ──────────────────────────────────────────
STAGE2_SYSTEM = """You are an NLP annotator. For EVERY User message in the conversation, output one JSON object.
Fields:
  turn (int, odd: 1,3,5...19), raw (exact text), expanded (self-contained rewrite resolving all pronouns/ellipsis using context; copy as-is if small-talk), topic_l1 (one of: Politics/Sports/Technology/Entertainment/Health/History/Geography/General), topic_l2 (sub-topic), named_entities (list), needs_expansion (bool)
Rules:
- Output ONLY a valid JSON array of exactly 10 objects. No markdown, no extra text.
- expanded must never be empty.
- named_entities must always be a list.
Example:
[{"turn":1,"raw":"who is the pm?","expanded":"who is the prime minister of india?","topic_l1":"Politics","topic_l2":"India","named_entities":["India"],"needs_expansion":false},
 {"turn":3,"raw":"what are his duties?","expanded":"what are Narendra Modi's duties as PM of India?","topic_l1":"Politics","topic_l2":"India","named_entities":["Narendra Modi","India"],"needs_expansion":true}]"""

def stage2_user_prompt(conv_text):
    return f"Annotate every User message. Return ONLY the JSON array.\n\n{conv_text}"


# ── STAGE 3: QUALITY CHECK ───────────────────────────────────────
STAGE3_SYSTEM = "Answer with exactly one word: VALID or INVALID."

def stage3_user_prompt(raw, expanded, l1, l2):
    return f"Raw: {raw}\nExpanded: {expanded}\nTopic: {l1}>{l2}\nIs expansion faithful and topic correct?"

print("✅ Compact prompts loaded.")

✅ Compact prompts loaded.


## 4. Helpers

In [7]:
import re
from datetime import timezone # Added to fix deprecation warning

def call_llm(system_prompt, user_prompt, temperature=0.8, max_tokens=3000):
    """Groq call with MAX_RETRIES retry logic."""
    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role":"system","content":system_prompt},
                    {"role":"user",  "content":user_prompt},
                ],
                temperature=temperature,
                max_tokens=max_tokens,
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            last_err = e
            print(f"  ⚠️  Attempt {attempt}/{MAX_RETRIES}: {type(e).__name__}: {e}")
            if attempt < MAX_RETRIES:
                print(f"  ⏳  Retry in {RETRY_DELAY}s...")
                time.sleep(RETRY_DELAY)
    raise RuntimeError(f"All {MAX_RETRIES} retries failed. Last: {last_err}")


def parse_conversation(raw):
    turns = []
    for line in raw.strip().split("\n"):
        line = line.strip()
        if not line:
            continue

        # Use Regex to safely catch "User:", "**User:**", "Assistant:", etc.
        match = re.match(r'^(?:\*\*)?(user|assistant)(?:\*\*)?\s*:\s*(.*)', line, re.IGNORECASE)
        if match:
            role = match.group(1).lower()
            text = match.group(2).strip()
            turns.append({"role": role, "text": text})

    # OVERSHOOT FIX: If the model generated 24 lines, slice it down to exactly 20
    if len(turns) >= 20:
        return turns[:20]

    return None


def parse_annotation(raw_json):
    text = re.sub(r'^```[a-z]*\s*','',raw_json.strip())
    text = re.sub(r'\s*```$','',text)
    try:
        data = json.loads(text)
        return data if isinstance(data, list) else None
    except json.JSONDecodeError:
        return None


def validate_sample(turns, annotation):
    if len(turns) != 20:
        return False, f"turn count={len(turns)}"
    expected = ["user" if i%2==0 else "assistant" for i in range(20)]
    if [t["role"] for t in turns] != expected:
        return False, "alternation violated"
    if not isinstance(annotation, list) or len(annotation) != 10:
        return False, f"annotation length={len(annotation) if annotation else 'None'}"
    required = {"turn","raw","expanded","topic_l1","topic_l2","named_entities","needs_expansion"}
    for i, ann in enumerate(annotation):
        if required - set(ann.keys()):
            return False, f"ann[{i}] missing fields"
        if not str(ann.get("expanded","")).strip():
            return False, f"ann[{i}] empty expanded"
        if not isinstance(ann.get("named_entities"), list):
            return False, f"ann[{i}] entities not list"
        if ann.get("topic_l1") not in ALLOWED_L1:
            return False, f"ann[{i}] bad topic_l1='{ann.get('topic_l1')}'"
    return True, "OK"


def build_row(conv_id, l1, l2, entity, turns, annotation):
    return {
        "conversation_id":  conv_id,
        "topic_l1":         l1,
        "topic_l2":         l2,
        "seed_entity":      entity,
        "turns":            json.dumps(turns, ensure_ascii=False),
        "user_queries":     json.dumps([a["raw"]            for a in annotation], ensure_ascii=False),
        "expanded_queries": json.dumps([a["expanded"]        for a in annotation], ensure_ascii=False),
        "named_entities":   json.dumps([a["named_entities"]  for a in annotation], ensure_ascii=False),
        "needs_expansion":  json.dumps([a["needs_expansion"] for a in annotation]),
        "created_at":       datetime.now(timezone.utc).isoformat(), # Fixed deprecation warning here
    }


CSV_FIELDS = ["conversation_id","topic_l1","topic_l2","seed_entity",
              "turns","user_queries","expanded_queries",
              "named_entities","needs_expansion","created_at"]

def append_to_csv(row):
    write_header = not CSV_PATH.exists()
    with open(CSV_PATH,"a",newline="",encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=CSV_FIELDS)
        if write_header: w.writeheader()
        w.writerow(row)

def flush_to_json(buffer):
    existing = []
    if JSON_PATH.exists():
        with open(JSON_PATH) as f:
            try: existing = json.load(f)
            except: pass
    parsed = []
    for row in buffer:
        r = dict(row)
        for field in ["turns","user_queries","expanded_queries","named_entities","needs_expansion"]:
            try: r[field] = json.loads(r[field])
            except: pass
        parsed.append(r)
    with open(JSON_PATH,"w",encoding="utf-8") as f:
        json.dump(existing + parsed, f, indent=2, ensure_ascii=False)

def save_checkpoint(ids):
    with open(CHECKPOINT_PATH,"w") as f:
        json.dump({"completed":list(ids),"saved_at":datetime.now(timezone.utc).isoformat()}, f) # Fixed deprecation warning here

def load_checkpoint():
    if CHECKPOINT_PATH.exists():
        with open(CHECKPOINT_PATH) as f:
            return set(json.load(f).get("completed",[]))
    return set()

print("✅ Helpers ready.")

✅ Helpers ready.


## 5. Main Generation Loop
```
seed → Stage1 (generate) → Stage2 (annotate) → Stage3 (validate) → save
                                                                  → checkpoint every 50
```

In [8]:
completed_ids = load_checkpoint()
print(f"📌 Checkpoint: {len(completed_ids)} samples already done.")

stats  = {"attempted":0,"stage1_failures":0,"stage2_failures":0,
          "validation_fails":0,"saved":len(completed_ids)}
buffer = []

pbar = tqdm(total=TARGET_SAMPLES, initial=len(completed_ids),
            desc="Generating", unit="sample")

for idx, (l1, l2, entity) in enumerate(ALL_SEEDS):

    conv_id = f"conv_{idx:04d}_{l1.lower()}_{l2.lower()}".replace(" ","_")

    if conv_id in completed_ids:
        continue
    if stats["saved"] >= TARGET_SAMPLES:
        tqdm.write(f"\n🎯 Target {TARGET_SAMPLES} reached.")
        break

    stats["attempted"] += 1

    # ── STAGE 1 ───────────────────────────────────────────────────
    try:
        raw_conv = call_llm(STAGE1_SYSTEM, stage1_user_prompt(l1,l2,entity),
                            temperature=0.85, max_tokens=2000)
    except RuntimeError as e:
        stats["stage1_failures"] += 1
        tqdm.write(f"❌ [{conv_id}] Stage1 failed: {e}")
        continue

    turns = parse_conversation(raw_conv)
    if turns is None:
        stats["stage1_failures"] += 1
        tqdm.write(f"⚠️  [{conv_id}] Stage1 parse error.")
        tqdm.write(f"--- RAW ---\n{raw_conv[:400]}\n---")
        continue

    conv_text = "\n".join(
        f"{'User' if t['role']=='user' else 'Assistant'}: {t['text']}"
        for t in turns
    )

    # ── STAGE 2 ───────────────────────────────────────────────────
    try:
        raw_ann = call_llm(STAGE2_SYSTEM, stage2_user_prompt(conv_text),
                           temperature=0.2, max_tokens=3000)
    except RuntimeError as e:
        stats["stage2_failures"] += 1
        tqdm.write(f"❌ [{conv_id}] Stage2 failed: {e}")
        continue

    annotation = parse_annotation(raw_ann)
    if annotation is None:
        stats["stage2_failures"] += 1
        tqdm.write(f"⚠️  [{conv_id}] Stage2 JSON parse failed.")
        continue

    # ── STAGE 3 ───────────────────────────────────────────────────
    is_valid, reason = validate_sample(turns, annotation)
    if not is_valid:
        stats["validation_fails"] += 1
        tqdm.write(f"⚠️  [{conv_id}] Validation failed: {reason}.")
        continue

    # ── SAVE ──────────────────────────────────────────────────────
    row = build_row(conv_id, l1, l2, entity, turns, annotation)
    append_to_csv(row)
    buffer.append(row)
    completed_ids.add(conv_id)
    stats["saved"] += 1
    pbar.update(1)

    # ── CHECKPOINT EVERY 50 ───────────────────────────────────────
    if stats["saved"] % SAVE_EVERY == 0:
        flush_to_json(buffer)
        save_checkpoint(completed_ids)
        buffer = []
        tqdm.write(f"💾 Checkpoint @ {stats['saved']} samples.")

    time.sleep(3.0)   # rate-limit buffer

pbar.close()

if buffer:
    flush_to_json(buffer)
    save_checkpoint(completed_ids)
    print(f"💾 Final flush: {len(buffer)} samples.")

print("\n" + "="*50)
print("✅ Done!")
print(f"   Saved             : {stats['saved']}")
print(f"   Stage1 failures   : {stats['stage1_failures']}")
print(f"   Stage2 failures   : {stats['stage2_failures']}")
print(f"   Validation fails  : {stats['validation_fails']}")
print(f"   CSV  → {CSV_PATH}")
print(f"   JSON → {JSON_PATH}")

📌 Checkpoint: 0 samples already done.


Generating:   0%|          | 0/500 [00:00<?, ?sample/s]

⚠️  [conv_0002_politics_india] Validation failed: ann[5] bad topic_l1='Health/History'.
⚠️  [conv_0034_politics_india] Validation failed: ann[5] bad topic_l1='Education'.
⚠️  [conv_0035_politics_india] Validation failed: ann[8] bad topic_l1='Economy'.
⚠️  [conv_0036_politics_india] Validation failed: ann[9] bad topic_l1='Economy'.
⚠️  [conv_0037_politics_uk] Validation failed: ann[8] bad topic_l1='Education'.
💾 Checkpoint @ 50 samples.
⚠️  [conv_0068_politics_india] Validation failed: ann[2] bad topic_l1='Environment/Health'.
⚠️  [conv_0070_politics_india] Validation failed: ann[2] bad topic_l1='Economy/Politics'.
⚠️  [conv_0075_politics_usa] Validation failed: ann[7] bad topic_l1='Economy'.
⚠️  [conv_0105_politics_uk] Validation failed: ann[9] bad topic_l1='Economy'.
💾 Checkpoint @ 100 samples.
⚠️  [conv_0109_politics_usa] Validation failed: ann[3] bad topic_l1='Environment'.
⚠️  [conv_0134_geography_cities] Validation failed: ann[8] bad topic_l1='Culture'.
⚠️  [conv_0140_politics_uk]

KeyboardInterrupt: 

## 6. Inspect & Verify

In [9]:
df = pd.read_csv(CSV_PATH)
print(f"Total rows : {len(df)}")
print("\n── topic_l1 ──")
print(df["topic_l1"].value_counts().to_string())
print("\n── topic_l2 (top 15) ──")
print(df["topic_l2"].value_counts().head(15).to_string())

Total rows : 237

── topic_l1 ──
topic_l1
Sports           64
Technology       48
Politics         44
Entertainment    26
Health           21
History          14
Geography        13
General           7

── topic_l2 (top 15) ──
topic_l2
Cricket          24
AI               24
India            22
USA              18
Space            16
Tennis           16
Football         16
Hollywood        13
Bollywood        13
UK               11
Olympics          8
Gadgets           8
Nutrition         7
Mental Health     7
Fitness           7


In [10]:
sample = df.iloc[0]
print(f"ID     : {sample['conversation_id']}")
print(f"Topic  : {sample['topic_l1']} > {sample['topic_l2']}")
print(f"Entity : {sample['seed_entity']}\n")
print("── CONVERSATION ─────────────────────────────")
for t in json.loads(sample["turns"]):
    role = "User      " if t["role"]=="user" else "Assistant "
    print(f"  {role}: {t['text']}")
print("\n── ANNOTATIONS ──────────────────────────────")
for i,(q,e,ents,ne) in enumerate(zip(
    json.loads(sample["user_queries"]),
    json.loads(sample["expanded_queries"]),
    json.loads(sample["named_entities"]),
    json.loads(sample["needs_expansion"]))):
    print(f"  Turn {2*i+1}:")
    print(f"    Raw      : {q}")
    print(f"    Expanded : {e}")
    print(f"    Entities : {ents}")
    print(f"    Expanded?: {ne}\n")

ID     : conv_0000_politics_india
Topic  : Politics > India
Entity : Narendra Modi

── CONVERSATION ─────────────────────────────
  User      : who is the current prime minister of india?
  Assistant : Narendra Modi has been Prime Minister since 2014.
  User      : what are his major achievements?
  Assistant : He implemented GST, demonetization, and launched Make in India.
  User      : his economic policies are quite controversial, aren't they?
  Assistant : Yes, some people criticize demonetization for its impact on small businesses.
  User      : what about his foreign policy?
  Assistant : He has strengthened ties with the US, Israel, and Japan through strategic partnerships.
  User      : brb, got a call
  Assistant : Sure!
  User      : back — what about his education background?
  Assistant : Narendra Modi studied at the University of Delhi and later at Gujarat University.
  User      : did he have a political career before becoming PM?
  Assistant : He was the Chief Minister o

In [ ]:
all_flags = []
for row in df["needs_expansion"]:
    all_flags.extend(json.loads(row))
total = len(all_flags)
exp   = sum(all_flags)
print(f"Total user turns       : {total}")
print(f"Needed expansion       : {exp:>5}  ({100*exp/total:.1f}%)")
print(f"Already self-contained : {total-exp:>5}  ({100*(total-exp)/total:.1f}%)")

## 7. Train / Val / Test Split

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["topic_l1"])
val_df, test_df   = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df["topic_l1"])

train_df.to_csv(OUTPUT_DIR/"train.csv", index=False)
val_df.to_csv(  OUTPUT_DIR/"val.csv",   index=False)
test_df.to_csv( OUTPUT_DIR/"test.csv",  index=False)

print(f"Train : {len(train_df):>4} rows (80%)")
print(f"Val   : {len(val_df):>4} rows (10%)")
print(f"Test  : {len(test_df):>4} rows (10%)")
print(f"\nFiles saved to {OUTPUT_DIR}/")